# ***AIF360 Plots***

# Plots — driven by `Data_Plot_AIF360.xlsx`

The AIF360 charts below no longer hard-code their numbers: each one reads a tab of
`Data_Plot.xlsx` at run time. Edit the spreadsheet, re-run the cell, and the figure updates.

**Expected tab layout** (all three tabs share it):

| | A | B | C ... |
|---|---|---|---|
| **row 1** | *(blank)* | *(blank)* | algorithm-category bands, merged |
| **row 2** | *(blank)* | method name | method name ... |
| **row 3+** | metric name | value | value ... |

Values are stored as fractions (`0.0892`) and plotted as percentages (`8.92`).
Adding a method = adding a column; adding a metric = adding a row and listing it in
`BAR_METRICS`. Nothing else needs to change.

`Disparate Impact` is read into the table but left off the bars by default — it is a ratio
around 1.0, not a percentage, and it is blank for most methods. See the last section for how
to plot it.


## Setup

In [3]:
import textwrap

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

EXCEL_PATH = 'Data_Plot_AIF360.xlsx'

# Metrics drawn as bars, in legend order. Must match the row labels in column A.
BAR_METRICS = [
    'Statistical Parity Difference',
    'Average Odds Difference',
    'Equal Opportunity Difference',
    'Theil Index',
    'Generalized Entropy Index',
]
ACCURACY_ROW = 'accuracy'

COLORS = ['#2ec4b6', '#f4845f', '#7b9cda', '#d966a0', '#8bc34a']
HATCHES = ['///', '///', 'xxx', '...', '///']

# Sanity check: this should list the tabs in the workbook.
print(pd.ExcelFile(EXCEL_PATH).sheet_names)

['AIF360-Gender', 'AIF360-Ethnicity', 'AIF360-Gender+Ethnicity']


### Reader — one tab in, tidy table out

In [5]:
def load_sheet(sheet_name, path=EXCEL_PATH):
    """Read one tab into a tidy DataFrame.

    Layout assumed: row 1 = algorithm-category bands (merged), row 2 = method
    names, column A = metric names, everything else numeric.

    Returns (table, categories):
      table      -- DataFrame indexed by metric name, columns = method names
      categories -- list of category labels, one per method (merged cells filled)
    """
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)

    methods = [str(m).strip() for m in raw.iloc[1, 1:]]
    categories = raw.iloc[0, 1:].ffill().fillna('').astype(str).str.strip().tolist()

    table = raw.iloc[2:, 1:].apply(pd.to_numeric, errors='coerce')
    table.index = [str(i).strip() for i in raw.iloc[2:, 0]]
    table.columns = methods
    return table, categories


# Peek at what comes back.
_table, _cats = load_sheet('AIF360-Gender')
display(_table)
print(list(zip(_table.columns, _cats)))

                               Original  Reweighing  ...  EGR + CEOP  RW + CEOP
accuracy                       0.594932    0.533256  ...    0.613102   0.614269
Statistical Parity Difference  0.019315    0.011162  ...    0.026320   0.032399
Average Odds Difference        0.019159    0.011108  ...    0.026508   0.032589
Equal Opportunity Difference   0.031583    0.022063  ...    0.033581   0.040506
Theil Index                    0.282339    0.369144  ...    0.302835   0.324454
Generalized Entropy Index      0.203615    0.267657  ...    0.216834   0.231286
Disparate Impact               1.039581    1.026375  ...    1.061728   1.084213

[7 rows x 12 columns]
[('Original', ''), ('Reweighing', 'Preprocessing Algorithms'), ('Disparate Impact Remover', 'Preprocessing Algorithms'), ('Learning Fair Representations', 'Preprocessing Algorithms'), ('Exponentiated Gradient Reduction', 'In-processing Algorithms'), ('Grid Search Reduction', 'In-processing Algorithms'), ('Equalized Odds Postprocessing'

### Plotter — same styling as before, values sourced from the tab

In [7]:
def plot_sheet(sheet_name, title=None, savefig=None, path=EXCEL_PATH,
               bar_metrics=BAR_METRICS, as_percent=True, figsize=None,
               show_categories=True, label_wrap=16):
    """Grouped bars (fairness metrics) + accuracy line, driven by one Excel tab."""
    table, categories = load_sheet(sheet_name, path)

    missing = [m for m in bar_metrics + [ACCURACY_ROW] if m not in table.index]
    if missing:
        raise KeyError(f"{sheet_name!r} is missing row(s): {missing}. "
                       f"Rows found: {list(table.index)}")

    scale = 100.0 if as_percent else 1.0
    data = {m: (table.loc[m].to_numpy(dtype=float) * scale) for m in bar_metrics}
    accuracy = table.loc[ACCURACY_ROW].to_numpy(dtype=float) * scale

    groups = list(table.columns)
    n_groups, n_bars = len(groups), len(bar_metrics)
    bar_w = min(0.15, 0.82 / n_bars)
    x = np.arange(n_groups)

    fig, ax1 = plt.subplots(figsize=figsize or (max(12, 1.7 * n_groups), 8))
    ax2 = ax1.twinx()

    offsets = np.linspace(-(n_bars - 1) / 2, (n_bars - 1) / 2, n_bars) * bar_w
    top = np.nanmax([np.nanmax(v) for v in data.values()])
    pad = 0.02 * top

    for i, (metric, values) in enumerate(data.items()):
        bars = ax1.bar(x + offsets[i], values, width=bar_w,
                       color=COLORS[i % len(COLORS)],
                       hatch=HATCHES[i % len(HATCHES)],
                       edgecolor='black', linewidth=0.6, label=metric, zorder=3)
        for bar, val in zip(bars, values):
            if np.isnan(val):
                continue
            ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + pad,
                     f'{val:.2f}', ha='center', va='bottom',
                     fontsize=6.5, rotation=90)

    ax2.plot(x, accuracy, color='black', marker='o', markersize=7,
             linewidth=1.8, zorder=5, label='Accuracy')
    for xi, acc in zip(x, accuracy):
        ax2.text(xi, acc + 1.0, f'{acc:.2f}%', ha='center', va='bottom',
                 fontsize=9, fontweight='bold')

    ax1.set_ylim(0, np.ceil((top * 1.25) / 10) * 10)
    ax2.set_ylim(0, 100 if as_percent else np.ceil(np.nanmax(accuracy) * 1.25))
    ax1.set_xlim(-0.5, n_groups - 0.5)

    ax1.set_xticks(x)
    ax1.set_xticklabels([textwrap.fill(g, label_wrap) for g in groups], fontsize=9)
    ax1.set_ylabel('Metrics Values (%)' if as_percent else 'Metric Value', fontsize=11)
    ax2.set_ylabel('Accuracy (%)' if as_percent else 'Accuracy', fontsize=11)

    ax1.yaxis.set_major_locator(plt.MultipleLocator(10))
    ax2.yaxis.set_major_locator(plt.MultipleLocator(10))
    ax1.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)

    if show_categories and any(categories):
        start = 0
        for i in range(1, n_groups + 1):
            if i == n_groups or categories[i] != categories[start]:
                if categories[start]:
                    ax1.text((start + i - 1) / 2, -0.12, categories[start],
                             transform=ax1.get_xaxis_transform(),
                             ha='center', va='top', fontsize=9,
                             style='italic', color='#444444')
                if i < n_groups:
                    ax1.axvline(i - 0.5, color='#999999', linestyle=':',
                                linewidth=1, zorder=1)
                start = i

    bar_handles = [mpatches.Patch(facecolor=COLORS[i % len(COLORS)],
                                  hatch=HATCHES[i % len(HATCHES)],
                                  edgecolor='black', label=metric)
                   for i, metric in enumerate(bar_metrics)]
    line_handle = plt.Line2D([0], [0], color='black', marker='o',
                             markersize=7, linewidth=1.8, label='Accuracy')
    ax1.legend(handles=bar_handles + [line_handle],
               loc='upper left', fontsize=9, framealpha=0.9)

    ax1.set_title(title or f'{sheet_name} — Fairness Metrics vs. Accuracy',
                  fontsize=13, pad=12)
    fig.tight_layout()
    if savefig:
        fig.savefig(savefig, dpi=150, bbox_inches='tight')
    plt.show()
    return table

## One figure per tab

Each cell below renders a single tab. Change the tab name, title, or output filename here;
the plotting logic stays in `plot_sheet`.

In [9]:
table = plot_sheet(
    'AIF360-Gender',
    title='AIF360 - Gender',
    savefig='Plots/AIF360-Gender.png',
)

In [10]:
table = plot_sheet(
    'AIF360-Ethnicity',
    title='AIF360 - Ethnicity',
    savefig='Plots/AIF360-Ethnicity.png',
)

In [11]:
table = plot_sheet(
    'AIF360-Gender+Ethnicity',
    title='AIF360 - Gender + Ethnicity',
    savefig='Plots/AIF360-Gender-Ethnicity.png',
)

### Or render every tab in the workbook at once

Picks up new tabs automatically — no cell to add.

In [13]:
for sheet in pd.ExcelFile(EXCEL_PATH).sheet_names:
    plot_sheet(
        sheet,
        title=f'AIF360 - {sheet.split("-", 1)[-1]}',
        savefig=f'{sheet.replace("+", "-")}.png',
    )

### Variations

`plot_sheet` takes keyword arguments for the common tweaks:

```python
# Plot Disparate Impact on its own (raw ratio, not a percentage)
plot_sheet('AIF360-Gender', bar_metrics=['Disparate Impact'], as_percent=False)

# Only the three parity metrics, wider figure
plot_sheet('AIF360-Ethnicity',
           bar_metrics=['Statistical Parity Difference',
                        'Average Odds Difference',
                        'Equal Opportunity Difference'],
           figsize=(20, 8))

# Drop the category bands under the x-axis
plot_sheet('AIF360-Gender', show_categories=False)
```

Comparing one metric across all three tabs:

```python
import pandas as pd
metric = 'Equal Opportunity Difference'
comparison = pd.DataFrame({
    sheet: load_sheet(sheet)[0].loc[metric] * 100
    for sheet in pd.ExcelFile(EXCEL_PATH).sheet_names
})
display(comparison.round(2))
```

# ***Fairlear Plots***

---
`Data_Plot_Fairlearn.xlsx` has no Fairlearn tabs, so these four cells still carry their original
hard-coded numbers and are otherwise unchanged. To drive them from the workbook too, add a tab
per chart in the same layout — metric names down column A, methods across row 2 — and call
`plot_sheet` with the age-group rows as `bar_metrics`.

In [17]:
"""
Fairlearn fairness plots — one figure per tab of Data_Plot_Fairlearn.xlsx.

Each tab is laid out like this:

        A        B           C                D                     ...
   1                         (blank)          Pre-processing        In-processing ...   <- category bands (merged)
   2             Algorithm   Original Model   Correlation Remover   ...                 <- x-axis labels
   3             Accuracy    0.8041           0.8547                ...                 <- dot/line plot
   4   Status    Normal      0.4772           0.4000                ...                 <- grouped bars
   5             Covid       0.1538           0.1764                ...
   6             Pneumonia   0.2363           0.6153                ...

The tab name carries both labels: "Gender+Ethnicity-Equalized Odds" becomes
the title "Gender+Ethnicity" and the y-axis label "Equalized Odds".

Usage — from a terminal:
    python plot_fairlearn.py                       # every tab -> ./plots/*.png
    python plot_fairlearn.py --excel path.xlsx --outdir figs
    python plot_fairlearn.py --sheet "Gender-Equalized Odds" --show

Usage — from a Jupyter notebook (do NOT call main(); argparse has no command
line to read there, so call these instead):
    plot_all()                                     # every tab, shown inline
    plot_sheet('Gender-Equalized Odds', show=True)
"""

import argparse
import os
import textwrap

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# ── Configuration ─────────────────────────────────────────────────────────────

EXCEL_PATH = 'Data_Plot_Fairlearn.xlsx'
OUT_DIR = 'Plots'

ACCURACY_ROW = 'Accuracy'      # row label (column B) holding the accuracy series
ALGORITHM_ROW = 'Algorithm'    # row label (column B) holding the x-axis names

# One colour + hatch per status, in legend order. Extra statuses cycle through.
COLORS = ['#2ec4b6', '#f4845f', '#7b9cda', '#d966a0', '#8bc34a']
HATCHES = ['///', '///', 'xxx', '...', '///']

# Tab names abbreviate a couple of metrics; expand them for the y-axis label.
YLABEL_OVERRIDES = {
    'Demo Parity': 'Demographic Parity',
}


# ── Reader — one tab in, tidy table out ───────────────────────────────────────

def load_sheet(sheet_name, path=EXCEL_PATH):
    """Read one tab into a tidy DataFrame.

    Returns (table, categories, title, ylabel):
      table      -- DataFrame indexed by row label ('Accuracy', 'Normal', ...),
                    columns = algorithm names
      categories -- one category label per algorithm (merged cells forward-filled)
      title      -- part of the tab name before the first '-'
      ylabel     -- part of the tab name after the first '-'
    """
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)

    # Column B holds the row labels, columns C onward hold the data.
    label_col, first_data_col = 1, 2

    header_row = raw.index[
        raw[label_col].astype(str).str.strip().str.lower() == ALGORITHM_ROW.lower()
    ]
    if len(header_row) == 0:
        raise KeyError(f"{sheet_name!r}: no {ALGORITHM_ROW!r} row found in column B.")
    header_row = header_row[0]

    algorithms = [str(a).strip() for a in raw.iloc[header_row, first_data_col:]]

    # Row above the header carries the merged category bands; forward-fill them
    # so every algorithm inherits the band it sits under.
    if header_row > 0:
        categories = (raw.iloc[header_row - 1, first_data_col:]
                      .ffill().fillna('').astype(str).str.strip().tolist())
    else:
        categories = [''] * len(algorithms)

    body = raw.iloc[header_row + 1:]
    table = body.iloc[:, first_data_col:].apply(pd.to_numeric, errors='coerce')
    table.index = [str(i).strip() for i in body.iloc[:, label_col]]
    table.columns = algorithms

    # Drop trailing blank rows that Excel leaves behind.
    table = table[table.index.notna() & (table.index != 'nan')]
    table = table.dropna(how='all')

    # "Gender+Ethnicity-Equalized Odds" -> ("Gender+Ethnicity", "Equalized Odds")
    title, _, ylabel = sheet_name.partition('-')
    title, ylabel = title.strip(), ylabel.strip()
    ylabel = YLABEL_OVERRIDES.get(ylabel, ylabel)

    return table, categories, title, ylabel


# ── Plotter ───────────────────────────────────────────────────────────────────

def plot_sheet(sheet_name, path=EXCEL_PATH, savefig=None, title=None, ylabel=None,
               as_percent=True, figsize=None, show_categories=True,
               label_wrap=16, show=False):
    """Grouped bars (one per status) + accuracy line, driven by one Excel tab."""
    table, categories, sheet_title, sheet_ylabel = load_sheet(sheet_name, path)

    title = title or sheet_title
    ylabel = ylabel or sheet_ylabel

    if ACCURACY_ROW not in table.index:
        raise KeyError(f"{sheet_name!r} is missing the {ACCURACY_ROW!r} row. "
                       f"Rows found: {list(table.index)}")

    # Everything that is not the accuracy row is a status drawn as bars.
    statuses = [r for r in table.index if r != ACCURACY_ROW]

    scale = 100.0 if as_percent else 1.0
    data = {s: table.loc[s].to_numpy(dtype=float) * scale for s in statuses}
    accuracy = table.loc[ACCURACY_ROW].to_numpy(dtype=float) * 100.0

    groups = list(table.columns)
    n_groups, n_bars = len(groups), len(statuses)
    bar_w = min(0.22, 0.82 / n_bars)
    x = np.arange(n_groups)

    fig, ax1 = plt.subplots(figsize=figsize or (max(12, 1.9 * n_groups), 8))
    ax2 = ax1.twinx()

    offsets = np.linspace(-(n_bars - 1) / 2, (n_bars - 1) / 2, n_bars) * bar_w
    top = np.nanmax([np.nanmax(v) for v in data.values()])
    pad = 0.02 * top

    # Bars: one cluster per algorithm, one bar per status.
    for i, (status, values) in enumerate(data.items()):
        bars = ax1.bar(x + offsets[i], values, width=bar_w,
                       color=COLORS[i % len(COLORS)],
                       hatch=HATCHES[i % len(HATCHES)],
                       edgecolor='black', linewidth=0.6, label=status, zorder=3)
        for bar, val in zip(bars, values):
            if np.isnan(val):
                continue
            ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + pad,
                     f'{val:.2f}', ha='center', va='bottom',
                     fontsize=7, rotation=90)

    # Accuracy: dots joined by a line, on the right-hand axis.
    ax2.plot(x, accuracy, color='black', marker='o', markersize=7,
             linewidth=1.8, zorder=5, label='Accuracy')
    for xi, acc in zip(x, accuracy):
        ax2.text(xi, acc + 1.0, f'{acc:.2f}%', ha='center', va='bottom',
                 fontsize=9, fontweight='bold')

    # Left axis: round the headroom up to a tick, and pick a tick step that
    # gives roughly 8 gridlines whatever the scale of the metric.
    step = next(s for s in (1, 2, 5, 10, 20, 25, 50, 100)
                if top * 1.30 / s <= 10)
    ax1.set_ylim(0, np.ceil((top * 1.30) / step) * step)
    ax2.set_ylim(0, 100)
    ax1.set_xlim(-0.5, n_groups - 0.5)

    ax1.set_xticks(x)
    ax1.set_xticklabels([textwrap.fill(g, label_wrap) for g in groups], fontsize=9)
    ax1.set_ylabel(f'{ylabel} (%)' if as_percent else ylabel, fontsize=11)
    ax2.set_ylabel('Accuracy (%)', fontsize=11)

    ax1.yaxis.set_major_locator(plt.MultipleLocator(step))
    ax2.yaxis.set_major_locator(plt.MultipleLocator(10))
    ax1.grid(axis='y', linestyle='--', alpha=0.4, zorder=0)

    # Category bands under the x-axis, with a dotted divider between them.
    if show_categories and any(categories):
        start = 0
        for i in range(1, n_groups + 1):
            if i == n_groups or categories[i] != categories[start]:
                if categories[start]:
                    ax1.text((start + i - 1) / 2, -0.16, categories[start],
                             transform=ax1.get_xaxis_transform(),
                             ha='center', va='top', fontsize=9,
                             style='italic', color='#444444')
                if i < n_groups:
                    ax1.axvline(i - 0.5, color='#999999', linestyle=':',
                                linewidth=1, zorder=1)
                start = i

    bar_handles = [mpatches.Patch(facecolor=COLORS[i % len(COLORS)],
                                  hatch=HATCHES[i % len(HATCHES)],
                                  edgecolor='black', label=status)
                   for i, status in enumerate(statuses)]
    line_handle = plt.Line2D([0], [0], color='black', marker='o',
                             markersize=7, linewidth=1.8, label='Accuracy')
    ax1.legend(handles=bar_handles + [line_handle],
               loc='upper left', fontsize=9, framealpha=0.9)

    ax1.set_title(f'Fairlearn - {title}', fontsize=13, pad=12)
    fig.tight_layout()

    if savefig:
        os.makedirs(os.path.dirname(savefig) or '.', exist_ok=True)
        fig.savefig(savefig, dpi=150, bbox_inches='tight')
        print(f'saved {savefig}')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return table


def safe_filename(sheet_name):
    return 'Fairlearn-'+sheet_name.strip().replace('+', '-').replace(' ', '_') + '.png'


# ── Entry point ───────────────────────────────────────────────────────────────

def in_notebook():
    """True when running inside a Jupyter/IPython kernel.

    Jupyter injects its own argv (-f .../kernel-xxxx.json), which argparse
    rejects, so under a kernel we skip the command line and use the defaults.
    """
    try:
        from IPython import get_ipython
        shell = get_ipython()
        return shell is not None and shell.__class__.__name__ == 'ZMQInteractiveShell'
    except Exception:
        return False


def plot_all(path=EXCEL_PATH, outdir=OUT_DIR, sheets=None, show=False):
    """Render every tab (or just `sheets`) — the notebook-friendly entry point."""
    sheets = sheets or pd.ExcelFile(path).sheet_names
    tables = {}
    for sheet in sheets:
        tables[sheet] = plot_sheet(
            sheet,
            path=path,
            savefig=os.path.join(outdir, safe_filename(sheet)) if outdir else None,
            show=show,
        )
    return tables


def main():
    notebook = in_notebook()

    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument('--excel', default=EXCEL_PATH, help='path to the workbook')
    ap.add_argument('--outdir', default=OUT_DIR, help='directory for the PNGs')
    ap.add_argument('--sheet', action='append',
                    help='plot only this tab (repeatable); default is every tab')
    ap.add_argument('--show', action='store_true', help='display each figure')

    # Under a kernel there is no user command line to read; take the defaults
    # and display the figures inline.
    args = ap.parse_args([]) if notebook else ap.parse_args()

    return plot_all(path=args.excel, outdir=args.outdir,
                    sheets=args.sheet, show=args.show or notebook)


if __name__ == '__main__':
    main()

saved Plots\Fairlearn-Gender-Demographic_Parity.png
saved Plots\Fairlearn-Gender-Equalized_Odds.png
saved Plots\Fairlearn-Ethnicity-Demographic_Parity.png
saved Plots\Fairlearn-Ethnicity-Equalized_Odds.png
saved Plots\Fairlearn-Gender-Ethnicity-Demo_Parity.png
saved Plots\Fairlearn-Gender-Ethnicity-Equalized_Odds.png
